# <font color="steelblue">230. Inferencia estadística sobre proporciones</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**


**Fecha última edición**: 10/05/2025

**Licencia**: <small><a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a><br /></small>

No olvides hacer una copia si deseas utilizarlo. Al usar estos contenidos, aceptas nuestros términos de uso y nuestra política de privacidad.

## <font color='steelblue'>Introducción</font>

**Descripción:** En este cuaderno se presentan los procesos de inferencia vinculados al análisis de una proporción poblacional o a la comparación de dos proporciones poblacionales.


## <font color='steelblue'> Objetivos de aprendizaje

* Conocer los procedimeintos de inferencia estadística para el estudio de una proporción poblacional y aplicarlos a situaciones reales.

* Conocer los procedimeintos de inferencia estadística para el estudio de dos proporciones poblacionales y aplicarlos a situaciones reales.



## <font color='steelblue'> Contenidos </font>

1. Introducción
1. Inferencia estadística sobre una proporción
1. Inferencia estadística para la comparación de dos proporciones poblacionales
1. Estudio de asociación de dos factores


## <font color='steelblue'> 1. Introducción

En el cuaderno anterior hemos estudiado los aspectos genéricos sobre los procedimientos de inferencia estadística, mientras que en este cuaderno nos centraremos en el análisis en profundidad de una proporción, o de la comparación de dos proporciones.

Antes que nada cargamos todos los módulos necesarios.

In [ ]:
# Cargamos módulos de análisis numérico
import numpy as np
import pandas as pd         # importamos pandas como pd
import math                 # importamos módulo para cáculos matemáticos

# Cargamos módulos de análisis gráficos
from plotnine import *      # importamos módulo para gráficos con ggplot
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
sns.set_theme(style="whitegrid")
%config InlineBackend.figure_format = 'retina'

from scipy import stats

Para ejemplificar el proceso de inferencia vamos a utilizar le banco de datos `stroke` que ya utilizamos en caudernos anteriores. Este estudio recoge información sobre la posibildiad de sufrir un ictus vinculado a un conjuto de variables que pueden o no influir en ello. Cargamos los datos:

In [ ]:
# Cargamos los datos 'stroke'
url = 'https://raw.githubusercontent.com/jmsocuellamos/DataPython/main/healthcare-dataset-stroke-data.csv'
stroke = pd.read_csv(url)

## <font color='steelblue'> 2. Inferencia estadística sobre una proporción

Dada una población de sujetos sobre la que se desea estudiar una característica de interés de tipo discreto, registrada con valores 0 y 1, donde el 1 indica que se cumple cierta condición, surge el parámetro poblacional $\theta$ que refleja la proporción de ocurrencias y que caracteriza la distribución de $X$ como:

$$X \sim Bernouilli(\theta)$$


Si extraemos una muestra de tamaño $n$ de la población anterior, tomamos como estimador puntual $\hat{\theta}$ la la proporción muestral observada cuya distribución en el muestreo viene dada por:

$$\hat{\theta} \sim N\left(\theta, \sqrt{\frac{\theta(1-\theta)}{n}} \right)$$


### <font color='steelblue'> Intervalo de confianza

Fijado el nivel de confianza $\alpha$ y utilizando la dstribución en el muestreo de $\hat{\theta}$, el intervalo de confianza al nivel $100(1-\alpha)\%$ viene dado por la expresión:

$$[\hat{\theta} - q_{1-\alpha/2} \cdot se(\hat{\theta}), \hat{\theta} + q_{1-\alpha/2}\cdot se(\hat{\theta})]$$

donde:

* $q_{1-\alpha/2}$ es el cuantil $1-\alpha/2$ de una distribución $N(0,1)$
* $se(\hat{\theta})$ es el error estándar de estimación que viene dado por:

$$\sqrt{\frac{\hat{\theta}(1-\hat{\theta})}{n}}$$

### <font color='steelblue'> Contraste de hipótesis

Para el establecimiento de un contraste de hipótesis sobre una proporción poblacional consideramos:

* El valor $\theta_0$ con el que deseamos comparar la proporción poblacional $\theta$.
* La significatividad con la que vamos a conluir nuestro procedimiento de contraste, $\alpha$.
* El estadístico que vamos a utilizar para resolver el contraste es

$$EC = \frac{\theta - \theta_0}{se(\hat{\theta})}$$

El estadístico de contraste refleja lo cerca o lejos que se encuentra el valor $\theta_0$ del valor poblacional, en términos del error muestral que estamos comentiendo. Además el valor del estadístico de contraste observado (necesario para obtener el p-valor) viene dado por:

$$EC_{obs} = \frac{\hat{\theta} - \theta_0}{se(\hat{\theta})}$$

que refleja la diferencia entre el valor muestral y el valor que deseamos contrastar.

A continuación se  muestran Los contrastes que podemos plantear.






#### <font color='steelblue'> Contraste bilateral

Comenzamos con el contraste habitual donde queremos ver si la proporción poblacional toma cierto valor. Aunque es muy habitual también es el que tiene menos sentido práctico ya que estamos siendo muy restrictivos en el posible valor que puede tomar la proporción poblacional. El contraste viene dado por:

$$\left\lbrace\begin{array}{l}
H_0: \theta = \theta_0\\
H_a: \theta \neq \theta_0
\end{array}\right.$$

#### <font color='steelblue'> Contraste unilateral a la izquierda

El contraste unilateral a la izquierda viene dado por:

$$\left\lbrace\begin{array}{l}
H_0: \theta \leq \theta_0\\
H_a: \theta > \theta_0
\end{array}\right.$$

#### <font color='steelblue'> Contraste unilateral a la derecha

El contraste unilateral a la izquierda viene dado por:

$$\left\lbrace\begin{array}{l}
H_0: \theta \geq \theta_0\\
H_a: \theta < \theta_0
\end{array}\right.$$

### <font color='steelblue'> Función para resolver el problema de inferencia

Para facilitar la realización del proceso inferencial sobre una proporción se define la función siguiente.

In [ ]:
def inferencia_una_proporcion(df, var, exito, alpha, val_hip, hipotesis):
  """
  Inferencia sobre una proporción poblacional: intervalo de confianza bilateral y el
  contraste de hipótesis que queramos

  Parámetros:
    - df: muestra de la población
    - var: variable (cualitativa) de interés
    - exito: nivel de la variable "var" que indica el éxito a analizar,
             la proporción que utilizaremos
    - alpha: nivel de significación
    - val_hip: valor que establecemos en el contrate de hipótesis, p0
    - hipotesis: tipo de hipótesis ('bilateral', 'uni izq', 'uni der')

  Devuelve:
    Un data frame con el intervalo de confianza, el estadístico de contraste,
    el p-valor y el resultado del contraste entre otras cosas en las columnas
  """
  # Preparación de datos
  variable_muestral = df[var]
  exito_muestral = variable_muestral == exito

  ## Estimadores muestrales

  # Tamaño de la muestra
  n_obs = len(df)
  # Número de exitos
  n_exitos = sum(exito_muestral)
  # Proporción muestral
  prop_muestral = n_exitos/n_obs
  # Error muestral
  error_muestral = ((prop_muestral*(1-prop_muestral))/n_obs)**0.5
  # Desviación típica de la distribución
  desv_tip_distr = ((val_hip*(1-val_hip))/n_obs)**0.5

  ## Proceso inferencial

  # Intervalo de confianza bilateral
  valor_critico_bilateral = stats.norm.isf(alpha/2)
  lim_inf = round(prop_muestral - valor_critico_bilateral*error_muestral,3)
  lim_sup = round(prop_muestral + valor_critico_bilateral*error_muestral,3)
  texto_IC = f'[{lim_inf}, {lim_sup}]'

  # Estadístico de contraste observado, valor crítico, planteamiento hipótesis nula utilizada y p-valor,
  # dependiendo del contraste que queramos

  # si queremos un contraste de hipótesis bilateral, a dos colas
  if hipotesis == 'bilateral':
    est_contraste = abs(prop_muestral - val_hip)/desv_tip_distr
    valor_critico = valor_critico_bilateral
    texto_hip = f'H0: p = {val_hip}'
    p_valor = stats.norm.sf(est_contraste)*2
  # si queremos un contraste de hipótesis unilateral a la izquierda
  elif hipotesis == 'uni izq':
    est_contraste = (prop_muestral - val_hip)/desv_tip_distr
    valor_critico = -stats.norm.isf(alpha)
    texto_hip = f'H0: p ≥ {val_hip}'
    p_valor = 1-stats.norm.sf(est_contraste)
  # si queremos un contraste de hipótesis unilateral a la derecha
  elif hipotesis == 'uni der':
    est_contraste = (prop_muestral - val_hip)/desv_tip_distr
    valor_critico = stats.norm.isf(alpha)
    texto_hip = f'H0: p ≤ {val_hip}'
    p_valor = stats.norm.sf(est_contraste)
  # si no indicamos bien el tipo de contraste, nos mostrará un error
  else:
    est_contraste = 0
    valor_critico = 0
    texto_hip = "Ponga un tipo de contraste válido"
    p_valor = 0

  # Conclusión del contraste
  if p_valor < alpha:
    conc = 'Rechazamos H0'
  else:
    conc = 'No rechazamos H0'

  # Especificación de los resultados que se devolverán en un diccionario
  data = {'Proporción muestral': [format(prop_muestral, '6.3f')],
          'IC': [texto_IC],
          'Hipótesis nula': [texto_hip],
          'Estadístico de contraste': [format(est_contraste, '6.3f')],
          'Valor crítico': [format(valor_critico, '6.3f')],
          'p-valor': [format(p_valor, '.4f')],
          'nivel de significación': [alpha],
          'Conclusión': [conc]}
  # Transformamos el diccionario en un DataFrame, con la fila que se llame "Inferencia de una proporción"
  respuesta = pd.DataFrame(data,index = ['Inferencia de una proporción'])
  return respuesta


### <font color='steelblue'> Ejemplos

A continuación mostramos diferentes ejemplos de inferencia sobre una proporción utilizando la función anterior y el dataframe `stroke`.

**Ejemplo 1.** Comenzaremos por investigar la posibilidad de que la proporción de sujetos que han sufrido un incidente cerebrovascular sea del 7.5%. En este caso "éxito" consiste en haber sufrido el accidente cerebrovascular y la variable que nos proporciona información al respecto es `stroke` que da valores 1 o 0 en función de si se ha sufrido o no este evento.

En base a la información proporcionada el contraste de hipótesis para resolver el problema de inferencia viene dado por:

$$\left\lbrace\begin{array}{l}
H_0: \theta = 0.075\\
H_a: \theta \neq 0.075
\end{array}\right.$$

Para resolverlo fijamos el valor de $\alpha = 0.05$ y ejecutanos la función anterior con los argumnetos que corresponden

In [ ]:
inferencia_una_proporcion( stroke, 'stroke', 1, 0.05, 0.075, "bilateral")

,Proporción muestral,IC,Hipótesis nula,Estadístico de contraste,Valor crítico,p-valor,nivel de significación,Conclusión
Inferencia de una proporción,0.049,"[0.043, 0.055]",H0: p = 0.075,7.130,1.960,0.0000,0.05,Rechazamos H0


Podemos ver como el porcentaje muestral (estimación puntual) se sitúa en el 4.87%, con un intervalo de confianza al 95% de $[4.3\%, 5.5\%]$, que no incluye al valor del 7.5% propuesto, lo que indica que la información muestral esta alejada del valor propuesto.

En términos del contraste de hipótesis podemos ver que el estadístico de contraste es muy superior al valor crítico resultando un p-valor significativo (inferior a 0.05) lo que nos permite concluir que hay evidencias para rechazar la hipótesis de que el porcentaje de sujetos que sufen un ictus es del 7.5%

**Ejemplo 2.** Buscamos ahora evidencias de que la proporción de sujetos con un incidente cerebrovascular sea superior al 7.5%. Esto significa plantear el contraste:

En base a la información proporcionada el contraste de hipótesis para resolver el problema de inferencia viene dado por:

$$\left\lbrace\begin{array}{l}
H_0: \theta \leq 0.075\\
H_a: \theta > 0.075
\end{array}\right.$$

Recordemos que para plantear el contraste debemos colocar en la hipótesis alternativa lo que deseamos estudiar, es decir, el suceso sobre el que deseamos obtener evidencias estadísticas.


In [ ]:
inferencia_una_proporcion( stroke, 'stroke', 1, 0.05, 0.075, "uni der")

,Proporción muestral,IC,Hipótesis nula,Estadístico de contraste,Valor crítico,p-valor,nivel de significación,Conclusión
Inferencia de una proporción,0.049,"[0.043, 0.055]",H0: p ≤ 0.075,-7.130,1.645,1.0000,0.05,No rechazamos H0


En este caso el estadístico de contarste es inferior al valor crítico obteniendo un p-valor superior a 0.05, lo que implica que tenemos evidencias estadísticas para no rechazar la hipótesis nula, es decir, que tenemos evidencias para concluir que la proporción de sujetos con unn incidente cerebrovascualr es menor o igual al 7.5%.

**Ejemplo 3.** Por último, estudiemos si podríamos admitir que la proporción de sujetos con un incidente cerebrovascular es inferior al 7.5%, es decir:

$$\left\lbrace\begin{array}{l}
H_0: \theta \geq 0.075\\
H_a: \theta < 0.075
\end{array}\right.$$


In [ ]:
inferencia_una_proporcion(stroke, 'stroke', 1, 0.05, 0.075, "uni izq")

,Proporción muestral,IC,Hipótesis nula,Estadístico de contraste,Valor crítico,p-valor,nivel de significación,Conclusión
Inferencia de una proporción,0.049,"[0.043, 0.055]",H0: p ≥ 0.075,-7.130,-1.645,0.0000,0.05,Rechazamos H0


En este caso tenemos evidencias estadísticas para rechazar la hipótesis nula, es decir, podemos concluir que la proporción de sujetos con un incidente cerebrovascular no es mayor o igual al 7.5%.

**Ejemplo 4.** Ajustemos algo más, ya que sabemos que en general tenemos evidencias de que la incidencia de incidentes cerebrovasculares es inferior al 7.5%. Podemos plantearnos ahora preguntas algo más complejas como por ejemplo:

1. ¿la proporción de hombres con un incidente cerebrovascular es inferior al 6%?
2. ¿se mantiene esa incidencia inferior al 6% para las mujeres?
3.  ¿y en general para los sujetos con edad superior a los 55?

En este caso en cada situación tenemos un conjunto de datos distintos y un contraste diferente. Vayamos paso a paso.

En la primera situación recordemos que tenemos una variable `gender` que identifica el sexo del enfermo con la categoria `Male`, de forma que el contraste de interés es:

$$\left\lbrace\begin{array}{l}
H_0: \theta_{Male} \geq 0.06\\
H_a: \theta_{Male} < 0.06
\end{array}\right.$$

Resolvamos el contraste sleccioanndo e primer luagr los datos sobre los que vamos a trabajar:

In [ ]:
# Del banco de datos orginal seleccionamos todos los hombres para resolver
df = stroke[stroke['gender'] == "Male"]
# Proceso inferencial
inferencia_una_proporcion(df, 'stroke', 1, 0.05, 0.06, "uni izq")

,Proporción muestral,IC,Hipótesis nula,Estadístico de contraste,Valor crítico,p-valor,nivel de significación,Conclusión
Inferencia de una proporción,0.051,"[0.042, 0.06]",H0: p ≥ 0.06,-1.730,-1.645,0.0418,0.05,Rechazamos H0


Podemos ver que la proporción estimada de hombres con incidente cerebrovascular es del 5.1% con un intervalo de confianza al 95% entre el 4.2% y el 6%. El p-valor del contraste es inferior a 0.05 lo que implica que debemos rechazar la hipótesis nula, por lo que podemos concluir que hay evidencias estadísticas que la proporción de hombres con ictus no es mayor o igual al 6%.

Analizemos el segundo caso donde el contraste de interés viene dado por:

$$\left\lbrace\begin{array}{l}
H_0: \theta_{Female} \geq 0.06\\
H_a: \theta_{Female} < 0.06
\end{array}\right.$$

In [ ]:
# Del banco de datos orginal seleccionamos todos las mujeres para resolver
df = stroke[stroke['gender'] == "Female"]
# Proceso inferencial
inferencia_una_proporcion(df, 'stroke', 1, 0.05, 0.06, "uni izq")

,Proporción muestral,IC,Hipótesis nula,Estadístico de contraste,Valor crítico,p-valor,nivel de significación,Conclusión
Inferencia de una proporción,0.047,"[0.04, 0.055]",H0: p ≥ 0.06,-2.974,-1.645,0.0015,0.05,Rechazamos H0


En este caso con una confianza del 95% la proporción de mujeres que han sufrido un ictus se sitúa entre el 4% y el 5.5%: El resultado del contraste (p-valor menor que 0.05) nos permite concluir igual que en el caso de los hombres, de foorma que el porcentaje de mujeres con ictus es inferior al 6%.

Por último, deseamos trabajar con los mayores de 55 años sobre el mismo tipo de contraste, es decir:

$$\left\lbrace\begin{array}{l}
H_0: \theta_{+55} \geq 0.06\\
H_a: \theta_{+55} < 0.06
\end{array}\right.$$

In [ ]:
# Del banco de datos orginal seleccionamos todos las mujeres para resolver
df = stroke[stroke['age'] >= 55]
# Proceso inferencial
inferencia_una_proporcion(df, 'stroke', 1, 0.05, 0.06, "uni izq")

,Proporción muestral,IC,Hipótesis nula,Estadístico de contraste,Valor crítico,p-valor,nivel de significación,Conclusión
Inferencia de una proporción,0.119,"[0.104, 0.134]",H0: p ≥ 0.06,10.508,-1.645,1.0000,0.05,No rechazamos H0


En este caso con una confianza del 95% el porcentaje de sujetos de más de 55 años que han sufrido un ictus se sitúa entre el 10.4% y el 13.4%, indicando que la incidencia es alta para este grupo de edad. Si nos fijamos en el contraste tenemos evidencias estadísticas para no rechazar la hipótesis nula (p-valor > 0.05), lo que implica que el porcentaje de incidencia de ictus en los mayores de 55 años es superior al 6%.

## <font color='steelblue'> 3. Inferencia estadística para la comparación de dos proporciones poblacionales

Dada dos poblacionesn de sujetos sobre las que se desea estudiar una característica de interés de tipo discreto, registrada con valores 0 y 1, donde el 1 indica que se cumple cierta condición, surgen los  parámetros poblacionales $\theta_1$ y $theta_2$ que reflejan la proporción de ocurrencias y que caracterizan la distribución de $X$ en cada pobalción como:

$$\text{Población 1: } X \sim Bernouilli(\theta_1)$$
$$\text{Población 2: } X \sim Bernouilli(\theta_2)$$

Para comparar ambas poblaciones utilizamos habitualmente el parámetro que viene dado por la diferencia de proporciones poblacionales:

$$\theta_1 - \theta_2$$

Si extraemos una muestra de tamaño $n_1$ de la población 1 y  tamaño $n_2$ de la población 2, y consideramos los estimadores puntuales $\hat{\theta_1}$ y  $\hat{\theta_2}$ (proporciones muestrales) ya vimos en el cuaderno anterior que la distribución en el muestreo apra la diferencia viene dada por:

$$\hat{\theta_1} - \hat{\theta_2} \sim N\left(\theta_1-\theta_2, \sqrt{\frac{\theta_1(1-\theta_1)}{n_1} + \frac{\theta_2(1-\theta_2)}{n_2}}\right)$$


Para facilitar la nomenclatura matemática consideramos $D=\theta_1 - \theta_2$, y $\hat{D} = \hat{\theta}_1 - \hat{\theta}_2$

### <font color='steelblue'> Intervalo de confianza

Fijado el nivel de confianza $\alpha$ y utilizando la dstribución en el muestreo de $\hat{D}$, el intervalo de confianza al nivel $100(1-\alpha)\%$ viene dado por la expresión:

$$[\hat{D} - q_{1-\alpha/2} \cdot se(\hat{D}), \hat{D} + q_{1-\alpha/2}\cdot se(\hat{D})]$$

donde:

* $q_{1-\alpha/2}$ es el cuantil $1-\alpha/2$ de una distribución $N(0,1)$
* $se(\hat{D})$ es el error estándar de estimación que viene dado por:

$$\sqrt{\frac{\hat{\theta}_1(1-\hat{\theta}_1)}{n_1} + \frac{\hat{\theta}_2(1-\hat{\theta}_2)}{n_2}}$$

### <font color='steelblue'> Contraste de hipótesis

Para el establecimiento de un contraste de hipótesis para la diferencia de proporciones poblacionales consideramos:

* La significatividad con la que vamos a conluir nuestro procedimiento de contraste, $\alpha$.
* El estadístico que vamos a utilizar para resolver el contraste es

$$EC = \frac{D}{se(\hat{D})}$$

El estadístico de contraste refleja lo cerca o lejos que se encuentran ambas pobalciones, en términos del error muestral que estamos comentiendo. Además el valor del estadístico de contraste observado (necesario para obtener el p-valor) viene dado por:

$$EC_{obs} = \frac{D_{obs}}{se(\hat{D})}$$

A continuación se  muestran Los contrastes habituales en esta situación.


#### <font color='steelblue'> Contraste bilateral

Comenzamos con el contraste habitual donde queremos ver si ambas proporciones pobalcionales pueden ser consideradas estad´sticamente iguales. El contraste viene dado por:

$$\left\lbrace\begin{array}{l}
H_0: \theta_1 = \theta_2\\
H_a: \theta_1 \neq \theta_2
\end{array}\right.$$

que es equivalente a

$$\left\lbrace\begin{array}{l}
H_0: D = 0\\
H_a: D \neq 0
\end{array}\right.$$

#### <font color='steelblue'> Contraste unilateral a la izquierda

El contraste unilateral a la izquierda en términos de la diferencia de proporciones viene dado por:

$$\left\lbrace\begin{array}{l}
H_0: D \leq 0\\
H_a: D > 0
\end{array}\right.$$

#### <font color='steelblue'> Contraste unilateral a la derecha

El contraste unilateral a la derecha en términos de la diferencia de proporciones viene dado por:

$$\left\lbrace\begin{array}{l}
H_0: D \geq 0\\
H_a: D < 0
\end{array}\right.$$

Todos los contarste anteriore se pueden genralizar considerando una valor $d_0$ para la diferencia de proporciones, de foorma que tendríamos los contrastes:

$$\left\lbrace\begin{array}{l}
H_0: D = d_0\\
H_a: D \neq d_0
\end{array}\right.$$

$$\left\lbrace\begin{array}{l}
H_0: D \leq d_0\\
H_a: D > d_0
\end{array}\right.$$

$$\left\lbrace\begin{array}{l}
H_0: D \geq d_0\\
H_a: D < d_0
\end{array}\right.$$

que respectivamente establecen que la diferencia es igual a $d_0$, superior a $d_0$ e inferior a $d_0$.

### <font color='steelblue'> Función para resolver el problema de inferencia

Com ya hicimos en el caso de una proporción vamos a definir una función que nos permite resolver el problema de inferencia de la comparación de dos proporciones poblacionales.

In [ ]:
def inferencia_dos_proporciones(df1, df2, var, exito, alpha, val_hip, hipotesis):
  """
  Inferencia sobre la diferencia entre dos proporciones poblacionales: intervalo
  de confianza bilateral y el contraste de hipótesis que indiquemos

  Parámetros:
    - df1: muestra de la población para el primer grupo
    - df2: muestra de la población para el segundo grupo
    - var: variable (cualitativa) de interés
    - exito: nivel de la variable "var" que indica el éxito a analizar,
             la proporción que utilizaremos
    - alpha: nivel de significación
    - val_hip: valor que establecemos en el contrate de hipótesis, d0
    - hipotesis: tipo de hipótesis ('bilateral', 'uni izq', 'uni der')

  Devuelve:
    Un data frame con el intervalo de confianza, el estadístico de contraste, el p-valor y el resultado del contraste entre otras cosas en las columnas
  """
  ## Preparación de datos
  variable_muestral_1 = df1[var]
  exito_muestral_1 = variable_muestral_1 == exito
  variable_muestral_2 = df2[var]
  exito_muestral_2 = variable_muestral_2 == exito

  ## Estimadores muestrales
  # Tamaño de la muestra
  n_obs = np.array([len(df1), len(df2)])
  # Número de exitos
  n_exitos = np.array([sum(exito_muestral_1), sum(exito_muestral_2)])
  # Proporción muestral
  prop_muestral = n_exitos/n_obs
  # Diferencia entre las proporciones muestrales
  dif_muestral = prop_muestral[0] - prop_muestral[1]
  # Error muestral
  error_muestral = (prop_muestral[0]*(1-prop_muestral[0])/n_obs[0] + prop_muestral[1]*(1-prop_muestral[1])/n_obs[1])**0.5
  # Estimación de la proporción global
  p_gorro = (n_obs[0]*prop_muestral[0] + n_obs[1]*prop_muestral[1])/sum(n_obs)
  # Desviación típica de la distribución
  desv_tip_distr = (p_gorro*(1-p_gorro)*(1/n_obs[0] + 1/n_obs[1]))**0.5

  # Selección del denominador para el cálculos del estadístico de contraste dependiendo del valor de d0
  if val_hip == 0:
    denominador = desv_tip_distr
  else:
    denominador = error_muestral

  ## Proceso inferencial
  # Intervalo de confianza bilateral
  valor_critico_bilateral = stats.norm.isf(alpha/2)
  lim_inf = round(dif_muestral - valor_critico_bilateral*error_muestral,3)
  lim_sup = round(dif_muestral + valor_critico_bilateral*error_muestral,3)
  texto_IC = f'[{lim_inf}, {lim_sup}]'

  # Estadístico de contraste observado, valor crítico, planteamiento hipótesis nula utilizada y p-valor, dependiendo del contraste que queramos
  # si queremos un contraste de hipótesis bilateral, a dos colas
  if hipotesis == 'bilateral':
    est_contraste = abs(dif_muestral - val_hip)/denominador
    valor_critico = valor_critico_bilateral
    texto_hip = f'H0: p1-p2 = {val_hip}'
    p_valor = stats.norm.sf(est_contraste)*2
  # si queremos un contraste de hipótesis unilateral a la izquierda
  elif hipotesis == 'uni izq':
    est_contraste = (dif_muestral - val_hip)/denominador
    valor_critico = -stats.norm.isf(alpha)
    texto_hip = f'H0: p1-p2 ≥ {val_hip}'
    p_valor = 1-stats.norm.sf(est_contraste)
  # si queremos un contraste de hipótesis unilateral a la derecha
  elif hipotesis == 'uni der':
    est_contraste = (dif_muestral - val_hip)/denominador
    valor_critico = stats.norm.isf(alpha)
    texto_hip = f'H0: p1-p2 ≤ {val_hip}'
    p_valor = stats.norm.sf(est_contraste)
  # si no indicamos bien el tipo de contraste, nos mostrará un error
  else:
    est_contraste = 0
    valor_critico = 0
    texto_hip = "Ponga un tipo de contraste válido"
    p_valor = 0

  # Conclusión del contraste
  if p_valor < alpha:
    conc = 'Rechazamos H0'
  else:
    conc = 'No rechazamos H0'

  # Especificación de los resultados que se devolverán en un diccionario
  data = {'Proporción muestral': [format(dif_muestral, '6.3f')],
          'IC': [texto_IC],
          'Hipótesis nula': [texto_hip],
          'Estadístico de contraste': [format(est_contraste, '6.3f')],
          'Valor crítico': [format(valor_critico, '6.3f')],
          'p-valor': [format(p_valor, '.4f')],
          'nivel de significación': [alpha],
          'Conclusión': [conc]}
  # Transformamos el diccionario en un DataFrame, con la fila que se llame "Inferencia de la diferencia entre dos proporciones"
  respuesta = pd.DataFrame(data,index = ['Inferencia de la diferencia entre dos proporciones'])
  return respuesta

### <font color='steelblue'> Ejemplos

A continuación mostramos diferentes ejemplos de uso de la función anterior.

**Ejemplo 1.** Para el banco de datos estamos interesados en saber si la incidencia de ictus  es igual en los hombres que en las mujeres. En este caso el contraste de interés es:

$$\left\lbrace\begin{array}{l}
H_0: D = 0\\
H_a: D \neq 0
\end{array}\right.$$

con $D = \theta_{Male}-\theta_{Female}.$

In [ ]:
# Valores de entrada de la función
# Tomamos como población 1 a los hombres y como 2 a las mujeres
df1 = stroke[stroke['gender'] == 'Male']
df2 = stroke[stroke['gender'] == 'Female']
# Resolvemos la inferencia
inferencia_dos_proporciones(df1, df2, 'stroke', 1, 0.05, 0, 'bilateral')

,Proporción muestral,IC,Hipótesis nula,Estadístico de contraste,Valor crítico,p-valor,nivel de significación,Conclusión
Inferencia de la diferencia entre dos proporciones,0.004,"[-0.008, 0.016]",H0: p1-p2 = 0,0.649,1.960,0.5163,0.05,No rechazamos H0


La diferencia estimada entre las proporciones de hombres y mujeres es de 4%, con un intervalo de confianza al 95% con extremos -8% y 16%, que incluye el valor 0. Esto significa que el valor 0, que habla sobre igualdad de las dos poblaciones, es plausible a un nivel de confianza del 95%. El contraste  de hipótesis proporciona un p-valor de 0.52, (p-valor > 0.05), que no permite rechazar la igualdad de las dos poblaciones, esto es, tenemos evidencias esatdísticas para concluir que las proporciones de incidencia para hombres y mujeres no son diferentes.

**Ejemplo 2.** Para el banco de datos estamos interesados en saber si la incidencia de ictus es igual en los sujetos de más de 55 años y los de menos de 55 años:

$$\left\lbrace\begin{array}{l}
H_0: D = 0\\
H_a: D \neq 0
\end{array}\right.$$

con $D = \theta_{+55}-\theta_{-55}.$

In [ ]:
# Valores de entrada de la función
# Tomamos como población 1 a los hombres y como 2 a las mujeres
df1 = stroke[stroke['age'] >= 55]
df2 = stroke[stroke['age'] <  55]
# Resolvemos la inferencia
inferencia_dos_proporciones(df1, df2, 'stroke', 1, 0.05, 0, 'bilateral')

,Proporción muestral,IC,Hipótesis nula,Estadístico de contraste,Valor crítico,p-valor,nivel de significación,Conclusión
Inferencia de la diferencia entre dos proporciones,0.108,"[0.093, 0.124]",H0: p1-p2 = 0,17.092,1.960,0.0000,0.05,Rechazamos H0


Ahora la conclusión es que claramente ambos grupos de sujetos son diferentes al respecto de la incidencia de enfermedades cerebrovasculares. El intervalo de confianza al 95% para la diferencia de proporciones es $[9.3\%,12.4\%]$, que no incluye al cero. El p-valor es claramente inferior a 0.05, por lo que se rechaza la igualdad de proporciones poblaciones, y se concluye que la incidencia de ictus en los mayores de 55 años es diferente de los menores de 55 años.

## <font color='steelblue'> 4. Estudio de asociación de dos factores

En el estudio de asociación entre dos factores se pretende investigar la posible asociación entre ellos, es decir, se desea conocer si ciertas combinaciones de niveles destacan respecto de otras respecto a que acumulan mayores frecuencias. Si no hay asociación, hablamos de independencia entre los factores, y en consecuencia no debe apreciarse ningún patrón de frecuencias en las distintas celdas que conforman la combinación de niveles de ambos factores. Este estudio se resuelve a través del **test de independencia**:

$$H_0: \text{Independencia entre factores}$$
$$H_a: \text{Asociación entre factores}$$

Este test compara las frecuencias observadas de cada casilla de la tabla de contingencia frente a las que esperaríamos obtener en caso de que ambos factores fueran independientes y no manifestaran ningún patrón de asociación. Para ello compara las frecuencias marginales por celdas, con las frecuencias por filas y columnas. Veamos en detalle cómo se resuelve a través de las frecuencias siguientes implicadas:

* $c_{ij}$ es la frecuencia observada para la combinación $i$ del factor 1 y el nivel $j$ del factor 2, calculada con respecto al total de casos,
* $c_{i.}$ es la frecuencia observada por fila, esto es, para cada nivel del factor 1,
* $c_{.j}$ es la frecuencia observada para la columna $j$, esto es, para cada nivel del factor 2,
* $\hat{c}_{ij}$ es la frecuencia estimada para la combinación $i$ del factor 1 y $j$ del factor 2; en el caso de independencia se calcularía como:

$$\hat{c}_{ij} = c_{i.}c_{.j}$$

Si los valores observados están muy alejados de los valores estimados diremos que se incumple la hipótesis de independencia, y por loa tanto existe relación entre los factores considerados. Para resolver este contraste utilizaremos el test chi-cuadrado de independencia. En caso de asociación podremos después determinar qué combinación de niveles contribuye en mayor medida a la asociación entre los factores.

Para resolver la inferencia relativa a asociación entre dos variables categóricas, utilizamos funciones del módulo `statsmodels`. Si **datos** representa un dataframe con las dos variables de interés y filas los registros disponibles, entonces

* **tabla=sm.stats.Table.from_data(datos)**, genera la tabla de contingencia base para toda la inferencia
* **tabla.table_orig** muestra las frecuencias observadas en cada uno de los cruces de las categorías
* **tabla.fittedvalues** da los valores estimados
* **tabla.test_nominal_association()** resuelve el contraste de hipótesis de independencia.
* **tabla.test_nominal_association().pvalue** da el p-valor de dicho contraste.
* **table.chi2_contribs** da las contribuciones de cada una de las combinaciones de categorías al estadístico chi-cuadrado y en consecuencia está aportando mayores diferencias/desviaciones respecto del resto entre los valores observados y estimados. Permite identificar dónde la asociación es mayor.

Las tablas de contingencia son una primera aproximación al estudio de asociación entre dos variables de tipo factor.

En primer lugar cargamos el módulo statmodels

In [ ]:
import statsmodels.api as sm

### <font color='steelblue'> Ejemplos

En estos ejemplos seguiremos utilizando el dataframe `stroke`. En este caso no estamos interesados únicamete en las proporciones poblacionales, sino si estas muestran cieto gardo de asociación cuando combinamos la información de dos factores.

**Ejemplo 1.** En primer lugar estamos interesados en saber si la incidencia de ictus puede estar relacionada o no con el sexo del sujeto. para ello en primer lugar seleccioanmos las variables del dataframe elimiando en este caso el nivel `Other` de la varaible `sex` que identifica los sujetos de los que desconocemos su sexo, y que por tanto no resultan útiles en el proceso de inferencia.

In [ ]:
# Seleccionamos hombres y mujeres, excluyendo los pacientes no clasificados
df = stroke[stroke['gender'] != 'Other']
# Preparamos los datos
data = df[['stroke', 'gender']]

Obteneos la tabla de contingencia asociada a ambas variables:

In [ ]:
# Obtenemos la tabla de contingencia
tabla = sm.stats.Table.from_data(data)
# Visualizamos las frecuencias observadas
tabla.table_orig

gender,Female,Male
stroke,,
0,2853,2007
1,141,108


Podemos ver ahora las frecuencias estimadas con los marginales por filas y columnas

In [ ]:
tabla.fittedvalues.round(1)

gender,Female,Male
stroke,,
0,2848.1,2011.9
1,145.9,103.1


A simple vista apreciamos pocas discrepancias entre los valores observados y los estimados. Resolvemos a continuación el contraste y visualizamos el p-valor del test de independencia.

In [ ]:
rslt = tabla.test_nominal_association()
format(rslt.pvalue, '.4f')

'0.5163'

Dado que el p-valor obtenido es superior a 0.05 no tenemos evidencia para rechazar la hipótesis de independencia entre los factores, esto es, no tenemos evidencias a favor de que exista asociación entre `stroke` y `sex`.

**Ejemplo 2.** Queremos estudiar ahora si la incidencia de ictus puede estar relacionada o no con la edad, cuando organizamos a los sujetos en mayores de 55 y menores de 55. En primer lugar debemos definir una nueva variable que recoja dicha agrupación.

In [ ]:
# Calculamos la variable categorizada:
stroke['age_cat'] = pd.cut(stroke.age, [0, 55, 100], include_lowest = True,
                           right = False,
                           labels = ['Less than 55', 'More than 55'])

Comenzamos con el análisis obteniendo en primer lugar la tabla de frecuencias observada y estimada.

In [ ]:
stroke.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke', 'age_cat'],
      dtype='object')

In [ ]:
# Preparamos los datos
data = stroke[['stroke', 'age_cat']]
# Frecuencias observadas
tabla = sm.stats.Table.from_data(data)
tabla.table_orig

age_cat,Less than 55,More than 55
stroke,,
0,3294,1567
1,37,212


Veamos la tabla de frecuencias estimada

In [ ]:
tabla.fittedvalues.round(1)

age_cat,Less than 55,More than 55
stroke,,
0,3168.7,1692.3
1,162.3,86.7


En este caso si parece observarse una diferencia entre frecuencias observadas y frecuencias estimadas (en caso de que los factores actuarán de forma independiente). Procedemos con el contraste de independencia:

In [ ]:
rslt = tabla.test_nominal_association()
format(rslt.pvalue, '.4f')

'0.0000'

Obtenemos evidencias estadísticas para rechazar independencia y en consecuencia concluir que existe asociación entre los grupos de edad definidos y el riesgo de sufrir un ictus.

Estudiamos a continuación la contribución al estadístico de contraste de cada celda de la tabla de contingencia, para identificar qué combinación o combinaciones muestran un patrón más predominante.

In [ ]:
# Tabla de contribuciones
tabla.chi2_contribs.round(2)

age_cat,Less than 55,More than 55
stroke,,
0,4.96,9.28
1,96.75,181.15


En base a las contribuciones, apreciamos que la incidencia de accidentes cerebrovasculares (`stroke=1`) es claramente superior en los sujetos mayores de 55 años. Queda de manifiesto la asociación entre ambas variables.

**Ejemplo 3.** Para finalizar queremos estudiar ahora si la incidencia de ictus puede estar relacionada o no con el hábito de fumar. Antes de analizar la tabla debemos eliminar la categoria `Unknown` de `smoking_status` ya que desconocemos el comportamiento de los sujetos.

In [ ]:
# Seleccionamos hombres y mujeres, excluyendo los pacientes no clasificados
df = stroke[stroke['smoking_status'] != 'Unknown']
# Preparamos los datos
data = df[['stroke', 'smoking_status']]
# Frecuencias observadas
tabla = sm.stats.Table.from_data(data)
tabla.table_orig

smoking_status,formerly smoked,never smoked,smokes
stroke,,,
0,815,1802,747
1,70,90,42


Obtenemos tabla de estimaciones

In [ ]:
tabla.fittedvalues.round(1)

smoking_status,formerly smoked,never smoked,smokes
stroke,,,
0,834.9,1784.8,744.3
1,50.1,107.2,44.7


Para finalizar realizamos el contraste de asociación

In [ ]:
rslt = tabla.test_nominal_association()
format(rslt.pvalue, '.4f')

'0.0033'

Dado que el p-valor es inferior a 0.05 podemos establecer que hay asociación entre el hábito fumador y la incidencia de ictus. Veamos las contribuciones:

In [ ]:
# Tabla de contribuciones
tabla.chi2_contribs.round(2)

smoking_status,formerly smoked,never smoked,smokes
stroke,,,
0,0.47,0.17,0.01
1,7.87,2.75,0.16


Lo resultados parecen indicar que los sujeos con mayor incidencia de ictus aon aquellos que fuman esporádicamente. Este resultado es algo llamativo porque pareceria más natural que la incidencia fuese superior en los fumadores habituales. Este resultado puede deberse a que hemos ignorado el carácter ordinal del factor `smokins_status`.

Para realizar este estudio debemos convertir el factor a factor ordinal y utilizar el método `test_ordinal_association()` sobre la tabla de contingencia obtenida. En priemr luagr convertimos el factor a ordinal

In [ ]:
# ordenamos los valores que aparecen en Fedu
cats_to_order = ['Unknown', 'never smoked', 'formerly smoked', 'smokes']
# las convertimos en etiquetas ordenadas, bajo un tipo 'category'
cats_dtype = pd.api.types.CategoricalDtype(cats_to_order, ordered = True)
# y asignamos el tipo de dichas etiquetas a la variable Fedu
stroke['smoking_status'] = stroke['smoking_status'].astype(cats_dtype)
stroke['smoking_status'].dtype

CategoricalDtype(categories=['Unknown', 'never smoked', 'formerly smoked', 'smokes'], ordered=True)

Obtenemos ahora la tabla de contingencia eliminando la categoria `Unknown`

In [ ]:
# Seleccionamos hombres y mujeres, excluyendo los pacientes no clasificados
df = stroke[stroke['smoking_status'] != "Unknown"]
# Preparamos los datos
data = df[['stroke', 'smoking_status']]
# Frecuencias observadas
tabla = sm.stats.Table.from_data(data)
tabla.table_orig

smoking_status,never smoked,formerly smoked,smokes
stroke,,,
0,1802,815,747
1,90,70,42


Resolvemos el contarste utilizanado la versión ordinal

In [ ]:
rslt = tabla.test_ordinal_association()
format(rslt.pvalue, '.4f')

'0.1954'

El p-valor no resulta significativo (superior a 0.05) indicando que no hay asociación entre los factores caundo tenemos en cuenta el orden.